# Crypto Portfolio Analytics implementation

Implementation for the matching report calls in `crypto_portfolio_analytics.ipynb`. Run the dashboard notebook to execute these methods in order.


In [ ]:
from datetime import datetime, timedelta, timezone
from getpass import getpass
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import requests
from IPython.display import display


def _signed_table(frame, formatters=None):
    """Format a table and color positive numeric values green and negatives red."""
    def sign_color(value):
        if isinstance(value, (int, float, np.number)) and not isinstance(value, (bool, np.bool_)):
            if pd.notna(value):
                if value > 0:
                    return 'color: #188038;'
                if value < 0:
                    return 'color: #c62828;'
        return ''
    return frame.style.format(formatters or {}, na_rep='N/A').map(sign_color)


class _TableStyleMixin:
    """Shared table formatting methods."""
    def show_table(self, frame, formatters=None):
        display(_signed_table(frame, formatters))


## Market data refresh

Dashboard call: `analysis.refresh_market_data()`. Validates CoinGecko credentials, refreshes saved USD prices and market-cap/volume history, and prints one concise status summary.


In [ ]:
class _MarketDataMixin:
    def refresh_market_data(self):
        api_plan = input('CoinGecko API plan (demo/pro): ').strip().lower()
        if api_plan not in {'demo', 'pro'}:
            raise ValueError("Enter 'demo' or 'pro' for your CoinGecko API plan.")
        api_key = getpass('CoinGecko API key (input hidden): ').strip()
        if not api_key:
            raise ValueError('A CoinGecko API key is required.')
        api_host = 'api.coingecko.com' if api_plan == 'demo' else 'pro-api.coingecko.com'
        api_header = 'x-cg-demo-api-key' if api_plan == 'demo' else 'x-cg-pro-api-key'
        headers = {api_header: api_key}
        api_base = f'https://{api_host}/api/v3'
        auth_params = {}

        def error_details(response):
            try:
                payload = response.json()
            except ValueError:
                return ''
            if isinstance(payload, dict) and payload.get('error'):
                return ' | CoinGecko error: {}'.format(payload['error'])
            status = payload.get('status', {}) if isinstance(payload, dict) else {}
            if not isinstance(status, dict):
                return ''
            error_code, message = status.get('error_code'), status.get('error_message')
            if error_code is None and not message:
                return ''
            return ' | CoinGecko code {}: {}'.format(error_code or 'n/a', message or 'no message')

        auth_check = requests.get(
            f'{api_base}/simple/price', params={'ids':'bitcoin','vs_currencies':'usd'},
            headers=headers, timeout=20,
        )
        if auth_check.status_code == 401:
            key_name = 'x_cg_demo_api_key' if api_plan == 'demo' else 'x_cg_pro_api_key'
            try:
                query_check = requests.get(
                    f'{api_base}/simple/price',
                    params={'ids':'bitcoin','vs_currencies':'usd',key_name:api_key}, timeout=20,
                )
            except requests.RequestException as exc:
                raise RuntimeError(
                    f'CoinGecko credential check failed ({type(exc).__name__}); the API key was not printed.'
                ) from None
            if query_check.ok:
                auth_params, headers = {key_name:api_key}, {}
            else:
                raise ValueError(
                    f'CoinGecko credential check failed (HTTP {query_check.status_code}). '
                    f'Confirm that the selected plan ({api_plan}) matches the key and that the key is active.'
                    f'{error_details(query_check)} The key was not printed.'
                )
        elif not auth_check.ok:
            raise RuntimeError(
                f'CoinGecko credential check failed (HTTP {auth_check.status_code}).'
                f'{error_details(auth_check)}'
            )

        data_dir = self.data_dir
        data_dir.mkdir(parents=True, exist_ok=True)
        price_file = data_dir / 'crypto_daily_prices_usd.csv'
        market_file = data_dir / 'crypto_market_data.csv'
        supported_coins = {'BTC':'bitcoin','VVV':'venice-token','HYPE':'hyperliquid'}
        requested_assets = list(dict.fromkeys(self.assets + [self.benchmark]))
        unknown = [asset for asset in requested_assets if asset not in supported_coins]
        if unknown:
            raise ValueError(f'No CoinGecko ID is configured for portfolio asset(s): {unknown}')
        coins = {asset:supported_coins[asset] for asset in requested_assets}
        fallback_start = datetime(2025, 10, 1, tzinfo=timezone.utc)
        end_date = datetime.now(timezone.utc)
        demo_earliest = end_date - timedelta(days=365)
        prices = (pd.read_csv(price_file, parse_dates=['date']) if price_file.exists()
                  else pd.DataFrame(columns=['date']))
        if not prices.empty:
            prices['date'] = pd.to_datetime(prices['date'], utc=True).dt.normalize()
            prices = prices.set_index('date').sort_index()
        else:
            prices = pd.DataFrame(index=pd.DatetimeIndex([], tz='UTC', name='date'))
        if market_file.exists():
            market = pd.read_csv(market_file, parse_dates=['date'])
            market['date'] = pd.to_datetime(market['date'], utc=True).dt.normalize()
            market = market.set_index('date').sort_index()
        else:
            market = pd.DataFrame(index=pd.DatetimeIndex([], tz='UTC', name='date'))

        def points_to_series(points, column):
            if not points:
                return pd.Series(dtype=float, name=column)
            frame = pd.DataFrame(points, columns=['timestamp_ms',column])
            frame['date'] = pd.to_datetime(frame.pop('timestamp_ms'),unit='ms',utc=True).dt.normalize()
            return pd.to_numeric(
                frame.drop_duplicates('date',keep='last').set_index('date')[column], errors='coerce'
            )

        updated = {}
        warnings = []
        for ticker, coin_id in coins.items():
            if ticker not in prices:
                prices[ticker] = pd.NA
            saved = pd.to_numeric(prices[ticker],errors='coerce').dropna()
            price_start = (saved.index.max()-pd.Timedelta(days=1)).to_pydatetime() if not saved.empty else fallback_start
            cap_col, volume_col = f'{ticker}_market_cap', f'{ticker}_volume'
            latest = [market[c].dropna().index.max() for c in (cap_col,volume_col)
                      if c in market and market[c].notna().any()]
            if len(latest)==2:
                market_start=(min(latest)-pd.Timedelta(days=1)).to_pydatetime()
            elif not saved.empty:
                market_start=saved.index.min().to_pydatetime()
            else:
                market_start=fallback_start
            start_date=min(price_start,market_start).replace(hour=0,minute=0,second=0,microsecond=0)
            if api_plan=='demo':
                start_date=max(start_date,demo_earliest)
            url=f'{api_base}/coins/{coin_id}/market_chart/range'
            params={'vs_currency':'usd','from':int(start_date.timestamp()),
                    'to':int(end_date.timestamp()),'interval':'daily',**auth_params}
            try:
                response=requests.get(url,params=params,headers=headers,timeout=30)
            except requests.RequestException as exc:
                raise RuntimeError(
                    f'{ticker}: CoinGecko request failed ({type(exc).__name__}); the API key was not printed.'
                ) from None
            if response.status_code==401 and not auth_params:
                key_name='x_cg_demo_api_key' if api_plan=='demo' else 'x_cg_pro_api_key'
                try:
                    query_response=requests.get(url,params={**params,key_name:api_key},timeout=30)
                except requests.RequestException as exc:
                    raise RuntimeError(
                        f'{ticker}: CoinGecko query-key retry failed ({type(exc).__name__}); the API key was not printed.'
                    ) from None
                if query_response.ok:
                    auth_params,headers={key_name:api_key},{}
                response=query_response
            if not response.ok:
                raise RuntimeError(
                    f'{ticker}: CoinGecko request failed (HTTP {response.status_code}).'
                    f'{error_details(response)} Check API-plan access and the requested date range.'
                )
            data=response.json()
            if 'prices' not in data:
                raise ValueError(f'{ticker}: CoinGecko response did not contain prices.')
            fetched=points_to_series(data['prices'],ticker)
            if not fetched.empty:
                prices=prices.reindex(prices.index.union(fetched.index)).sort_index()
                prices.loc[fetched.index,ticker]=fetched
            else:
                warnings.append(f'{ticker}: CoinGecko returned no new price observations.')
            updated[ticker]=len(fetched)
            for response_key,column in [('market_caps',cap_col),('total_volumes',volume_col)]:
                series=points_to_series(data.get(response_key,[]),column)
                if series.empty:
                    warnings.append(f'{ticker}: no {response_key} history was returned.')
                    continue
                market=market.reindex(market.index.union(series.index)).sort_index()
                market.loc[series.index,column]=series
        prices=prices.sort_index()
        prices.to_csv(price_file,date_format='%Y-%m-%dT%H:%M:%SZ')
        market=market.sort_index()
        market.index.name='date'
        market.reset_index().to_csv(market_file,index=False,date_format='%Y-%m-%dT%H:%M:%SZ')
        self.updated_observations=updated
        self.refresh_warnings=warnings
        self.api_plan=api_plan
        self.market_data=market
        message='Refreshed saved prices and market data: '+', '.join(
            f'{ticker} {count:,} days' for ticker,count in updated.items()
        )+'.'
        if api_plan=='demo':
            message+=' Demo history requests are limited to the latest 365 days; older saved rows were retained.'
        print(message)
        for warning in warnings:
            print(warning)


## Price history

Dashboard call: `analysis.load_price_history()`. Load and validate the saved CoinGecko price series.


In [ ]:
class _PriceHistoryMixin(_TableStyleMixin):
    def load_price_history(self):
        price_file=self.data_dir/'crypto_daily_prices_usd.csv'
        if not price_file.exists():
            raise FileNotFoundError(f'Missing saved price file: {price_file.resolve()}')
        prices=pd.read_csv(price_file,parse_dates=['date']).set_index('date').sort_index()
        prices.index=pd.to_datetime(prices.index,utc=True).normalize()
        required=list(dict.fromkeys(self.assets+[self.benchmark]))
        missing=[ticker for ticker in required if ticker not in prices.columns]
        if missing:
            raise ValueError(f'Missing saved price columns: {missing}')
        self.prices=prices[required].apply(pd.to_numeric,errors='coerce').sort_index()


## Daily returns and correlations

Dashboard call: `analysis.calculate_returns()`. Build the complete daily-return sample and its Pearson correlation matrix.


In [ ]:
class _ReturnAnalysisMixin(_TableStyleMixin):
    def calculate_returns(self):
        returns=self.prices.pct_change(fill_method=None).replace([np.inf,-np.inf],np.nan)
        required=list(dict.fromkeys(self.assets+[self.benchmark]))
        common=returns.dropna(subset=required)
        if len(common)<30:
            raise ValueError(f'Only {len(common)} common daily returns; need at least 30.')
        self.returns=returns
        self.common_returns=common
        self.correlation=common[self.assets].corr(method='pearson')


## Correlation heatmap

Dashboard call: `analysis.plot_correlations()`. Display the correlation matrix as an interactive Plotly heatmap.


In [ ]:
class _CorrelationPlotMixin(_TableStyleMixin):
    def plot_correlations(self):
        corr=self.correlation
        fig=go.Figure(go.Heatmap(
            z=corr.to_numpy(),x=corr.columns.tolist(),y=corr.index.tolist(),
            zmin=-1,zmax=1,colorscale='RdBu',reversescale=True,
            text=np.round(corr.to_numpy(),2),texttemplate='%{text:.2f}',
            hovertemplate='%{y} vs %{x}: %{z:.3f}<extra></extra>',
            colorbar={'title':'Correlation'},
        ))
        fig.update_layout(title='Daily return correlations',template='plotly_white',
                          width=850,height=700)
        fig.update_yaxes(autorange='reversed')
        fig.show()


## Constrained full-Kelly portfolio

Dashboard call: `analysis.optimize_portfolio()`. Optimize the risky asset weights, keep any unallocated share as cash, and display weights with signed green/red formatting.


In [ ]:
def _project_weights(weights, lower, upper, budget=1.0):
    projected=np.clip(weights,lower,upper)
    if projected.sum()<=budget:
        return projected
    left=np.min(weights-upper)-1.0
    right=np.max(weights-lower)+1.0
    for _ in range(100):
        shift=(left+right)/2.0
        projected=np.clip(weights-shift,lower,upper)
        if projected.sum()>budget:
            left=shift
        else:
            right=shift
    return np.clip(weights-right,lower,upper)


def _optimize_full_kelly(returns, lower, upper, cash_daily_return, max_iterations=20000):
    def objective(weights):
        portfolio_returns=returns@weights+(1.0-weights.sum())*cash_daily_return
        if np.any(portfolio_returns<=-1.0):
            return float('inf')
        return -np.log1p(portfolio_returns).mean()
    def gradient(weights):
        portfolio_returns=returns@weights+(1.0-weights.sum())*cash_daily_return
        marginal_returns=returns-cash_daily_return
        return -(marginal_returns/(1.0+portfolio_returns[:,None])).mean(axis=0)
    weights=_project_weights(np.full(returns.shape[1],1.0/returns.shape[1]),lower,upper)
    step_size=1.0
    for _ in range(max_iterations):
        current=objective(weights)
        grad=gradient(weights)
        trial=step_size
        for _ in range(100):
            candidate=_project_weights(weights-trial*grad,lower,upper)
            direction=candidate-weights
            candidate_value=objective(candidate)
            bound=current+grad@direction+(direction@direction)/(2.0*trial)
            if candidate_value<=bound+1e-14:
                break
            trial*=0.5
        else:
            raise RuntimeError('Kelly optimizer could not find a stable step.')
        if np.max(np.abs(direction))<1e-10:
            return candidate
        weights=candidate
        step_size=min(trial*1.2,1e6)
    raise RuntimeError('Kelly optimizer did not converge within the iteration limit.')


class _KellyMixin(_TableStyleMixin):
    def optimize_portfolio(self):
        self.min_weight=0.02
        self.max_weight=0.15
        self.risk_free_annual=0.05
        self.trading_days_per_year=365
        self.risk_free_daily=(1+self.risk_free_annual)**(1/self.trading_days_per_year)-1
        returns=self.common_returns[self.assets].to_numpy(dtype=float)
        weights=_optimize_full_kelly(
            returns,self.min_weight,self.max_weight,self.risk_free_daily
        )
        self.full_kelly=pd.Series(weights,index=self.assets,name='weight').sort_values(ascending=False)
        self.full_portfolio=pd.concat([
            self.full_kelly,pd.Series({'Cash':1.0-self.full_kelly.sum()},name='weight')
        ])
        print('Constrained full-Kelly weights, sorted by coin weight (%)')
        self.show_table((100*self.full_portfolio).rename('Weight (%)').to_frame(),
                        {'Weight (%)':'{:.2f}'})


## Portfolio performance and risk metrics

Dashboard call: `analysis.analyze_performance()`. Show Plotly portfolio-vs-BTC growth and styled trailing-return, drawdown, and risk tables.


In [ ]:
class _PerformanceMixin(_TableStyleMixin):
    def analyze_performance(self):
        benchmark='BTC'
        if benchmark not in self.common_returns.columns:
            raise ValueError(f'Benchmark {benchmark} is not in the return data.')
        weights=self.full_kelly.reindex(self.assets).to_numpy()
        portfolio_returns=pd.Series(
            self.common_returns[self.assets].to_numpy()@weights
            +self.full_portfolio.loc['Cash']*self.risk_free_daily,
            index=self.common_returns.index,name='Portfolio',
        )
        benchmark_returns=self.common_returns[benchmark]
        excess=portfolio_returns-self.risk_free_daily
        benchmark_excess=benchmark_returns-self.risk_free_daily
        beta=portfolio_returns.cov(benchmark_returns)/benchmark_returns.var(ddof=1)
        annual_return=portfolio_returns.mean()*self.trading_days_per_year
        daily_vol=portfolio_returns.std(ddof=1)
        annual_vol=daily_vol*np.sqrt(self.trading_days_per_year)
        var95=max(0.0,-portfolio_returns.quantile(0.05))
        var99=max(0.0,-portfolio_returns.quantile(0.01))
        sharpe=(excess.mean()/daily_vol)*np.sqrt(self.trading_days_per_year)
        alpha=(excess.mean()-beta*benchmark_excess.mean())*self.trading_days_per_year
        initial_value=100.0
        initial_date=self.common_returns.index[0]-pd.Timedelta(days=1)
        portfolio_value=pd.concat([
            pd.Series([initial_value],index=pd.DatetimeIndex([initial_date])),
            initial_value*(1+portfolio_returns).cumprod(),
        ])
        btc_value=pd.concat([
            pd.Series([initial_value],index=pd.DatetimeIndex([initial_date])),
            initial_value*(1+benchmark_returns).cumprod(),
        ])
        portfolio_value.name='Portfolio value ($)'
        btc_value.name='BTC value ($)'
        fig=go.Figure()
        fig.add_trace(go.Scatter(x=portfolio_value.index,y=portfolio_value,name='Constrained full-Kelly',
                                 line={'color':'#2563eb','width':2}))
        fig.add_trace(go.Scatter(x=btc_value.index,y=btc_value,name='BTC',
                                 line={'color':'#F7931A','width':2}))
        fig.update_layout(title='Since-inception value: constrained full-Kelly vs. BTC',
                          xaxis_title='Date',yaxis_title='Value of $100 invested',
                          template='plotly_white',hovermode='x unified')
        fig.show()
        as_of=portfolio_value.index[-1]
        def trailing_return(days=None,months=None):
            target=as_of-(pd.DateOffset(months=months) if months is not None else pd.Timedelta(days=days))
            prior=portfolio_value.loc[portfolio_value.index<=target]
            if prior.empty:
                return np.nan
            return portfolio_value.iloc[-1]/prior.iloc[-1]-1
        period_returns=pd.Series({
            '1D':trailing_return(days=1),'7D':trailing_return(days=7),
            'M (1 month)':trailing_return(months=1),'Q (3 months)':trailing_return(months=3),
            'Since Inception':portfolio_value.iloc[-1]/100-1,
        },name='Return')
        self.show_table(period_returns.to_frame(),{'Return':'{:.2%}'})
        daily_summary=pd.Series({
            'Average daily return':portfolio_returns.mean(),
            'Largest daily gain':portfolio_returns.max(),
            'Largest daily loss':portfolio_returns.min(),
        },name='Daily return')
        self.show_table(daily_summary.to_frame(),{'Daily return':'{:.2%}'})
        peak=portfolio_value.cummax()
        drawdown=portfolio_value/peak-1
        drawdown_dollars=portfolio_value-peak
        max_drawdown=drawdown.min()
        investment_summary=pd.DataFrame([{
            'All-Time High ($)':portfolio_value.max(),
            'Minimum Portfolio Value ($)':portfolio_value.min(),
            'Max Drawdown ($)':drawdown_dollars.min(),
            'Max Drawdown (%)':max_drawdown,
        }],index=['Since inception'])
        self.show_table(investment_summary,{
            'All-Time High ($)':'${:,.2f}','Minimum Portfolio Value ($)':'${:,.2f}',
            'Max Drawdown ($)':lambda v:f'-${abs(v):,.2f}' if v<0 else f'${v:,.2f}',
            'Max Drawdown (%)':'{:.2%}',
        })
        risk_free_annual=(1+self.risk_free_daily)**self.trading_days_per_year-1
        metrics=pd.DataFrame([{
            'Sharpe':sharpe,'Beta':beta,'Alpha':alpha,'Expected Return':annual_return,
            'Hist Vol (daily)':daily_vol,'Hist Vol (annualized)':annual_vol,
            '1D Hist VaR 95%':var95,'VaR 95% ($ / $100)':var95*100,
            '1D Hist VaR 99%':var99,'VaR 99% ($ / $100)':var99*100,
            'Risk-Free Rate':risk_free_annual,'Max Drawdown':max_drawdown,
        }],index=['Constrained full-Kelly'])
        self.show_table(metrics,{
            'Sharpe':'{:.3f}','Beta':'{:.3f}','Alpha':'{:.2%}',
            'Expected Return':'{:.2%}','Hist Vol (daily)':'{:.2%}',
            'Hist Vol (annualized)':'{:.2%}','1D Hist VaR 95%':'{:.2%}',
            'VaR 95% ($ / $100)':'${:,.2f}','1D Hist VaR 99%':'{:.2%}',
            'VaR 99% ($ / $100)':'${:,.2f}','Risk-Free Rate':'{:.2%}',
            'Max Drawdown':'{:.2%}',
        })
        self.portfolio_daily_returns=portfolio_returns
        self.portfolio_value=portfolio_value
        self.btc_value=btc_value
        self.performance_metrics=metrics
        self.beta=beta
        self.max_drawdown=max_drawdown
        self.portfolio_period=(self.common_returns.index.min(),self.common_returns.index.max())


## CoinGecko factor model

Dashboard call: `analysis.fit_factor_model()`. Fit excess returns against market and available cross-sectional factors, then display the factor estimates and fit summary.


In [ ]:
def _long_short_factor(signal, asset_returns, long_low=False):
    factor=pd.Series(np.nan,index=asset_returns.index,dtype=float)
    for date in asset_returns.index:
        available=signal.loc[date].dropna().index.intersection(asset_returns.columns)
        if len(available)<4:
            continue
        ranked=signal.loc[date,available].sort_values()
        group_size=len(ranked)//2
        if long_low:
            long_assets,short_assets=ranked.index[:group_size],ranked.index[-group_size:]
        else:
            long_assets,short_assets=ranked.index[-group_size:],ranked.index[:group_size]
        daily_returns=asset_returns.loc[date]
        factor.loc[date]=daily_returns[long_assets].mean()-daily_returns[short_assets].mean()
    return factor


def _normalize_daily_index(data):
    normalized=data.copy()
    normalized.index=pd.to_datetime(normalized.index,utc=True).normalize()
    return normalized.groupby(level=0,sort=True).last()


class _FactorModelMixin(_TableStyleMixin):
    def fit_factor_model(self):
        market_file=self.data_dir/'crypto_market_data.csv'
        if not market_file.exists():
            raise FileNotFoundError(f'Missing {market_file}. Run the CoinGecko refresh cell first.')
        market=pd.read_csv(market_file,parse_dates=['date'])
        market['date']=pd.to_datetime(market['date'],utc=True).dt.normalize()
        market=market.set_index('date').sort_index()
        market_caps=pd.DataFrame(index=self.common_returns.index)
        volumes=pd.DataFrame(index=self.common_returns.index)
        for asset in list(dict.fromkeys(self.assets+[self.benchmark])):
            cap_col,volume_col=f'{asset}_market_cap',f'{asset}_volume'
            if cap_col not in market or volume_col not in market:
                raise ValueError(f'Market-cap/volume history is missing for {asset}; rerun the CoinGecko refresh cell.')
            market_caps[asset]=pd.to_numeric(market[cap_col],errors='coerce').reindex(self.common_returns.index).ffill(limit=2)
            volumes[asset]=pd.to_numeric(market[volume_col],errors='coerce').reindex(self.common_returns.index).ffill(limit=2)
        alt_assets=[asset for asset in self.assets if asset!='BTC']
        lagged_caps=market_caps.shift(1)
        alt_weights=lagged_caps[alt_assets].div(lagged_caps[alt_assets].sum(axis=1),axis=0)
        alt_return=(self.common_returns[alt_assets]*alt_weights).sum(axis=1,min_count=len(alt_assets))
        factors=pd.DataFrame(index=self.common_returns.index)
        factors['BTC market excess']=self.common_returns['BTC']-self.risk_free_daily
        if alt_assets:
            factors['Alt market - BTC']=alt_return-self.common_returns['BTC']
        factor_note=None
        if len(alt_assets)>=4:
            momentum=self.prices[alt_assets].pct_change(30).shift(1).reindex(self.common_returns.index)
            turnover=volumes[alt_assets].div(market_caps[alt_assets].where(market_caps[alt_assets]>0))
            lagged_turnover=turnover.shift(1).rolling(30,min_periods=20).mean()
            factors['Small - large']=_long_short_factor(lagged_caps[alt_assets],self.common_returns[alt_assets],long_low=True)
            factors['Momentum']=_long_short_factor(momentum,self.common_returns[alt_assets])
            factors['High - low turnover']=_long_short_factor(lagged_turnover,self.common_returns[alt_assets])
        else:
            factor_note=(f'Size, momentum, and turnover spreads omitted: {len(alt_assets)} selected altcoins; '
                         'at least 4 are needed.')
        factors=_normalize_daily_index(factors)
        portfolio_excess=_normalize_daily_index(
            (self.portfolio_daily_returns-self.risk_free_daily).rename('Portfolio excess')
        )
        model_data=pd.concat([portfolio_excess,factors],axis=1).replace([np.inf,-np.inf],np.nan).dropna()
        if len(model_data)<60:
            raise ValueError(
                f'Factor model has only {len(model_data)} complete aligned daily observations; at least 60 are required. '
                'Confirm that the portfolio and factor cells use the same saved price history.'
            )
        names=list(factors.columns)
        y=model_data['Portfolio excess'].to_numpy(dtype=float)
        X=np.column_stack([np.ones(len(model_data)),model_data[names].to_numpy(dtype=float)])
        coefficients,_,rank,_=np.linalg.lstsq(X,y,rcond=None)
        if rank<X.shape[1]:
            raise ValueError('Factor model is rank-deficient; check factor correlations or history coverage.')
        residuals=y-X@coefficients
        r_squared=1.0-(residuals@residuals)/np.sum((y-y.mean())**2)
        adjusted=1.0-(1.0-r_squared)*(len(y)-1)/(len(y)-X.shape[1])
        scores=X*residuals[:,None]
        meat=scores.T@scores
        lags=min(7,len(y)-1)
        for lag in range(1,lags+1):
            weight=1.0-lag/(lags+1.0)
            cross=scores[lag:].T@scores[:-lag]
            meat+=weight*(cross+cross.T)
        bread=np.linalg.pinv(X.T@X)
        covariance=bread@meat@bread
        standard_errors=np.sqrt(np.maximum(np.diag(covariance),0.0))
        t_stats=np.divide(coefficients,standard_errors,out=np.full_like(coefficients,np.nan),where=standard_errors>0)
        annual_premiums=model_data[names].mean()*self.trading_days_per_year
        annual_contributions=pd.Series(coefficients[1:],index=names)*annual_premiums
        summary=pd.DataFrame({
            'Beta':coefficients[1:],'HAC t-stat':t_stats[1:],
            'Factor premium (%/yr)':annual_premiums,
            'Portfolio contribution (%/yr)':annual_contributions,
        },index=names)
        summary.loc['Alpha']=[np.nan,t_stats[0],np.nan,coefficients[0]*self.trading_days_per_year]
        fit=pd.Series({
            'Observations':len(model_data),'R-squared':r_squared,'Adjusted R-squared':adjusted,
            'Annual alpha':coefficients[0]*self.trading_days_per_year,
            'BTC beta (factor model)':coefficients[1],'BTC beta (single factor)':self.beta,
        },name='Factor model fit')
        print(f'Factor model: R²={r_squared:.1%} | adjusted R²={adjusted:.1%} | '
              f'annual alpha={coefficients[0]*self.trading_days_per_year:.2%} | n={len(model_data)} days')
        print(f'Single-factor BTC beta={self.beta:.3f} | factor-model BTC beta={summary.loc["BTC market excess","Beta"]:.3f}')
        if factor_note:
            print(factor_note)
        self.show_table(summary,{
            'Beta':'{:.3f}','HAC t-stat':'{:.2f}',
            'Factor premium (%/yr)':'{:.2%}',
            'Portfolio contribution (%/yr)':'{:.2%}',
        })
        self.show_table(fit.to_frame())
        self.factor_returns=factors
        self.factor_model_summary=summary
        self.factor_model_fit=fit


## Save analysis outputs

Dashboard call: `analysis.save_outputs()`. Write the refreshed source data, portfolio results, daily returns, and factor estimates to `data/`.


In [ ]:
class _OutputMixin:
    def save_outputs(self):
        data=self.data_dir
        data.mkdir(parents=True,exist_ok=True)
        self.correlation.to_csv(data/'crypto_return_correlation.csv')
        self.full_portfolio.to_csv(data/'crypto_kelly_full_weights.csv',header=True)
        self.full_portfolio.to_csv(data/'crypto_kelly_weights.csv',header=True)
        self.performance_metrics.to_csv(data/'crypto_portfolio_metrics.csv')
        self.portfolio_daily_returns.to_csv(data/'crypto_portfolio_daily_returns.csv',header=True)
        self.prices.to_csv(data/'crypto_daily_prices_usd.csv')
        self.market_data.sort_index().rename_axis('date').reset_index().to_csv(
            data/'crypto_market_data.csv',index=False,date_format='%Y-%m-%dT%H:%M:%SZ'
        )
        self.common_returns.to_csv(data/'crypto_daily_returns.csv')
        self.factor_model_summary.to_csv(data/'crypto_portfolio_factor_model.csv')
        self.factor_model_fit.to_csv(data/'crypto_portfolio_factor_fit.csv')
        self.factor_returns.to_csv(data/'crypto_factor_returns.csv')
        print('Saved portfolio, factor, market, and return outputs to data/.')


## Analytics object

The report notebook creates one `CryptoPortfolioAnalytics` instance and calls the named methods above.


In [ ]:
class CryptoPortfolioAnalytics(
    _MarketDataMixin, _PriceHistoryMixin, _ReturnAnalysisMixin, _CorrelationPlotMixin,
    _KellyMixin, _PerformanceMixin,
    _FactorModelMixin, _OutputMixin, _TableStyleMixin,
):
    def __init__(self, data_dir, assets):
        self.data_dir=Path(data_dir)
        self.assets=list(assets)
        self.benchmark='BTC'
        if not self.assets or len(set(self.assets)) != len(self.assets):
            raise ValueError('assets must be a non-empty list with no duplicates.')
        self.data_dir.mkdir(parents=True,exist_ok=True)
